In [1]:
#implementing parts of GPT Transformer
import torch
import torch.nn as nn

In [2]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

In [3]:
class LayerNorm(nn.Module):
    def __init__(self,emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = torch.ones(emb_dim)
        self.shift = torch.zeros(emb_dim)

    def forward(self,x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x-mean)/torch.sqrt(var+self.eps)
        return self.scale + norm_x + self.shift

class FeedForward(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.layers=  nn.Sequential(
            nn.Linear(cfg['emb_dim'], 4*cfg['emb_dim']), 
            nn.GELU(),
            nn.Linear(4*cfg['emb_dim'],cfg['emb_dim'])

        )

    def forward(self,x):
        return self.layers(x)




In [5]:
from attention_mechanism import MultiHeadAttention


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], 
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm = LayerNorm(cfg['emb_dim'])
        self.dropout = nn.Dropout(cfg['drop_rate'])

    def forward(self,x):
        #shortcut connection for attention block
        shortcut = x
        x = self.norm(x)
        x = self.att(x)
        x = self.dropout(x)
        x = x + shortcut

        return x
        
    

In [8]:
#testing the config 

torch.manual_seed(143)
x = torch.rand(2,6,768)
model = TransformerBlock(GPT_CONFIG_124M)
output = model(x)

print(output.shape), x.shape

torch.Size([2, 6, 768])


(None, torch.Size([2, 6, 768]))

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        